In [ ]:
import pandas as pd

In [ ]:
train_df = pd.read_csv("Train.csv");
test_df = pd.read_csv("Test.csv");

train_df.dropna(inplace=True)

slang = pd.read_csv("slang.csv");

In [ ]:
test_df

In [ ]:
ans_sb = pd.DataFrame()

import re

def sb3(prop):
    ans = 0;
    for w in prop.split():
        if(w[0] == '#' and ("vaccine" in w or "vaccinated" in w or "vacc" in w)): ans += 1
    return ans;

ans_sb["sb1"] = test_df["safe_text"].apply(lambda x: len(x.split()));
ans_sb["sb2"] = test_df["safe_text"].apply(lambda x: x.lower());
ans_sb["sb3"] = test_df["safe_text"].apply(lambda x: sb3(x));
ans_sb["sb6"] = test_df["safe_text"].apply(lambda x: len(re.findall(r"([0-9]{4}[0-9]{3}[0-9]{3})", x)))

In [ ]:
ans_sb

In [ ]:
ans_sb.describe()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

test_bow = CountVectorizer().fit_transform(test_df["safe_text"].values);
test_tfidf = TfidfVectorizer().fit_transform(test_df["safe_text"].values);

In [ ]:
print(test_bow.shape)
print(test_tfidf.shape)

In [ ]:
map_slang = {}

def add_to_map(row):
    abv = row["Abbreviations"]
    mean = row["Text"]

    map_slang[abv] = mean;
    return row;

slang = slang.apply(add_to_map, axis=1);


In [ ]:
train_df

In [ ]:
map_slang["mr"] = "mister"
map_slang["mr."] = "mister"
map_slang["w"] = "with"


In [ ]:
map_slang

In [ ]:
train_text = train_df["safe_text"].values
test_text = test_df["safe_text"].values
train_labels = train_df["label"].values

In [ ]:
processed_text = []
import re

def custom_text_preproecessor(props):
    global processed_text
    processed_text = []

    proc_props = props

    # null_chars = ['{{USERNAME}}', '{{URL}}', '\'', '(', ')', '[', ']', '{', '}', '/', '\\', '|', '-', '_', '+', '=', '*', '^', '&', '%', '$', '#', '@', '!', '~', '`', ';', ':', ',', '.', '<', '>', '?', '"']

    for i, prop in enumerate(proc_props):
        prop.replace("\n", " ")
        prop.replace("\t", " ")
        prop.replace("\r", " ")
        prop.replace("#", "")
        prop.replace("@", "")
        RE = re.compile(r"\s+")

        new_prop = ""

        for ch in prop:
            if(ch.isalnum() or ch.isspace()):
                new_prop += ch 

        lower_case = new_prop.lower();
        lower_case = RE.sub(" ", lower_case).strip()

        new_prop = ""

        for w in lower_case.split():
            if(w in map_slang): new_prop = new_prop + map_slang[w] + " "
            elif(len(w) > 3 or w == "not" or w == "yes" or w == "Not"): new_prop = new_prop + w + " "

        proc_props[i] = new_prop;


    processed_text = proc_props

custom_text_preproecessor(train_text)

train_text = processed_text

custom_text_preproecessor(test_text)

test_text = processed_text

In [ ]:
train_text


In [ ]:
from nltk.stem import SnowballStemmer

stem = SnowballStemmer(language="english");

def stem_text(text):
    new_text = []
    
    for prop in text:
        new_prop = " ".join([stem.stem(w) for w in prop.split()])
        new_text.append(new_prop)

    return new_text

train_text = stem_text(train_text)
test_text = stem_text(test_text)

In [ ]:
def remove_non_ascii(text):
    new_text = []

    for t in text:
        new_text.append(t.encode("ascii", "ignore").decode())

    return new_text

train_text = remove_non_ascii(train_text)
test_text = remove_non_ascii(test_text)

In [ ]:
def remove_multiple_spaces(text):
    new_text = []

    for t in text:
        new_text.append(" ".join(t.split()))

    return new_text

train_text = remove_multiple_spaces(train_text)
test_text = remove_multiple_spaces(test_text)

In [ ]:
def replace_slang(text):
    new_text = []

    for t in text:
        words = t.split(' ');

        # print(words);

        new_p = "";

        for w in words:
            # print(w);
            if(w in map_slang): new_p += map_slang[w] + " ";
            else: new_p += w + " ";
        
        new_text.append(new_p);

    return new_text

train_text = replace_slang(train_text)
test_text = replace_slang(test_text)

In [ ]:
train_text

In [ ]:
def remove_words_with_fs_ch_digit(text):
    new_text = []

    def has_numbers(inputString):
        return any(char.isdigit() for char in inputString)


    for t in text:
        new_text.append(" ".join([word for word in t.split(' ') if not has_numbers(word) and len(word) > 2]))

    return new_text

train_text = remove_words_with_fs_ch_digit(train_text)
test_text = remove_words_with_fs_ch_digit(test_text)

In [ ]:
import re
def replace_ch_repetitions_with_ch(text):
    new_text = []

    for t in text:
        new_text.append(" ".join([re.sub(r'([a-z])\1+', r'\1', word) for word in t.split(' ')]))

    return new_text

train_text = replace_ch_repetitions_with_ch(train_text)
test_text = replace_ch_repetitions_with_ch(test_text)

In [ ]:
train_text

In [ ]:
from sklearn.feature_extraction.text import *


cv = TfidfVectorizer().fit(train_text);

train_text = cv.transform(train_text);
test_text = cv.transform(test_text)

In [ ]:
from sklearn.decomposition import *
import matplotlib.pyplot as plt

points = PCA(2).fit_transform(train_text)

plt.scatter(points[:, 0], points[:, 1], c=train_labels)

In [ ]:
from sklearn.naive_bayes import GaussianNB
import catboost
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier(verbose=1, n_estimators=1000).fit(train_text, train_labels)

In [ ]:
pred = model.predict_proba(test_text)

In [ ]:
f = open("ans.csv", 'w');

f.write("tweet_id,label\n");

ids = test_df["tweet_id"];

for i, p in enumerate(pred):
    f.write(str(ids[i]) + "," + str(-p[0] + p[2]) + "\n")
f.close()